# Toxicity Detection for Turkic Languages

Detecting toxic, offensive, or profane language is a critical component of
content moderation, brand safety filters, and safe-messaging platforms.
For Turkic languages, the problem is particularly challenging because:

- Most languages are **low-resource**: large annotated toxicity corpora
  simply do not exist for Tatar, Kyrgyz, Turkmen, Uyghur, or Bashkir.
- Agglutinative morphology makes **token matching fragile**: a toxic root
  may appear with dozens of different suffixes, all unseen by a simple
  word-list filter.
- Scripts differ across languages, so a single monolingual approach
  cannot be directly reused.

## Dataset: FLORES-200 Toxicity Word Lists (Toxicity-200)

Facebook Research maintains the **Toxicity-200** dataset—a curated word list
of toxic terms for all 200 NLLB-200 languages, grouped into four categories:

- Frequently used profanities
- Insults, hate speech, and demeaning language
- Pornographic terms
- Terms for body parts associated with sexual activity

The word lists are distributed as password-protected ZIP archives.

**Download URL:** `https://tinyurl.com/NLLB200TWL`
**Extraction password:** `tL4nLLb`
**File naming:** `<BCP47-code>_twl.zip`, one per language.

FLORES-200 Turkic languages covered:

| TurkicNLP ISO | FLORES BCP-47 | Language |
|:---:|:---:|:---|
| tur | tur_Latn | Turkish |\n| aze | azj_Latn | Azerbaijani |\n| kaz | kaz_Cyrl | Kazakh |\n| kir | kir_Cyrl | Kyrgyz |\n| tat | tat_Cyrl | Tatar |\n| tuk | tuk_Latn | Turkmen |\n| uig | uig_Arab | Uyghur |\n| uzb | uzn_Latn | Uzbek |

## Approaches Covered in This Notebook

| Section | Approach | Strengths | Weaknesses |
|---------|---------|-----------|------------|
| A | Token-based (keyword matching) | Fast, interpretable, no training needed | Misses morphological variants, context-blind |
| B | Embedding-based (per-language) | Captures context, handles morphology | Needs labelled examples per language |
| C | Unified multilingual classifier | One model for all languages, cross-lingual transfer | Requires combining training data |

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import os, zipfile, math, pathlib, urllib.request
import turkicnlp
from turkicnlp import Pipeline

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report, accuracy_score
    from sklearn.model_selection import train_test_split
except ImportError:
    raise SystemExit("pip install scikit-learn")

FLORES_LANGS = [
    ("tur", "tur_Latn", "Turkish"),
    ("aze", "azj_Latn", "Azerbaijani"),
    ("kaz", "kaz_Cyrl", "Kazakh"),
    ("kir", "kir_Cyrl", "Kyrgyz"),
    ("tat", "tat_Cyrl", "Tatar"),
    ("tuk", "tuk_Latn", "Turkmen"),
    ("uig", "uig_Arab", "Uyghur"),
    ("uzb", "uzn_Latn", "Uzbek"),
]

TWL_DIR = pathlib.Path("toxicity_wordlists")
TWL_DIR.mkdir(exist_ok=True)

## Section A — Token-based Toxicity Detection

### A.1 Download and Load Toxicity Word Lists

We download the ZIP archive for each language, extract using the published
password, and load the word list into a Python set for O(1) lookup.

> **Note:** the files contain toxic language; treat them as sensitive data
> and do not display their contents directly in shared environments.

In [ ]:
# ---------------------------------------------------------------------------
# Download helper — fetches and extracts one language word list
# ---------------------------------------------------------------------------
TWL_URL      = "https://tinyurl.com/NLLB200TWL/{code}_twl.zip"
TWL_PASSWORD = b"tL4nLLb"

def download_wordlist(nllb_code: str) -> set:
    """Download and extract the toxicity word list for a FLORES-200 language."""
    zip_path = TWL_DIR / f"{nllb_code}_twl.zip"
    txt_path = TWL_DIR / f"{nllb_code}_twl.txt"

    if not txt_path.exists():
        if not zip_path.exists():
            url = f"https://tinyurl.com/NLLB200TWL"
            # The actual individual-file URL pattern; adjust if the tinyurl
            # redirects to a bulk download page requiring manual steps.
            print(f"  Downloading word list for {nllb_code}...")
            try:
                urllib.request.urlretrieve(
                    f"https://github.com/facebookresearch/flores/raw/main/"
                    f"toxicity/{nllb_code}_twl.zip",
                    zip_path,
                )
            except Exception as e:
                print(f"  Auto-download failed ({e}).")
                print(f"  Please download manually from {url}")
                print(f"  and place '{nllb_code}_twl.zip' in '{TWL_DIR}/'")
                return set()

        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
            for name in names:
                zf.extract(name, TWL_DIR, pwd=TWL_PASSWORD)
        # Rename extracted file to consistent name if needed
        extracted = list(TWL_DIR.glob(f"*{nllb_code}*"))
        if extracted and extracted[0] != txt_path:
            extracted[0].rename(txt_path)

    words = set()
    if txt_path.exists():
        with open(txt_path, encoding="utf-8") as f:
            for line in f:
                w = line.strip().lower()
                if w:
                    words.add(w)
    return words

# Load word lists (downloads automatically if not cached)
wordlists = {}
for iso, nllb_code, name in FLORES_LANGS:
    wl = download_wordlist(nllb_code)
    wordlists[nllb_code] = wl
    print(f"  {name:<15}: {len(wl):>5} toxic terms loaded")

### A.2 Token-based Classifier

In [ ]:
def tokenize_simple(text: str) -> list:
    """Whitespace + punctuation split, lowercase."""
    import re
    return re.findall(r"[\w']+", text.lower())

def token_classifier(text: str, wordlist: set) -> dict:
    """
    Classify text as toxic/safe using keyword matching.
    Returns a dict with 'label', 'score', and 'matched_count'.
    """
    tokens  = tokenize_simple(text)
    matches = [t for t in tokens if t in wordlist]
    score   = len(matches) / max(len(tokens), 1)
    return {
        "label":         "toxic" if matches else "safe",
        "score":         score,
        "matched_count": len(matches),
    }

# ---- Demo: Turkish token-based classifier ----
turkish_wl = wordlists.get("tur_Latn", set())

demo_sentences = [
    "Bugün hava çok güzel, parkta yürüyüş yaptım.",
    "Bu toplantı çok verimli geçti.",
    "Seni hiç sevmiyorum, berbat birisin.",        # mild negative
    "Bu film harika, kesinlikle izleyin.",
]

print(f"{'Sentence':<50} {'Label':>6} {'Score':>6}")
print("-" * 65)
for sent in demo_sentences:
    result = token_classifier(sent, turkish_wl)
    print(f"{sent[:49]:<50} {result['label']:>6} {result['score']:>6.3f}")

### A.3 Evaluating the Token-based Approach

A key weakness of keyword matching is that it misses morphologically inflected forms of toxic roots. In Turkish, for example, a toxic root can appear with dozens of case, number, and tense suffixes—all distinct tokens from the base form in the word list. The embedding-based approach in Section B addresses this limitation.

In [ ]:
# Demonstrate the morphological coverage gap
toxic_root_examples = [
    # (surface form, expected: toxic?)
    # Assume 'kötü' (bad/mean) is in the word list as a mild example
    ("kötüsün", True),    # 'you are bad' — suffix -sün
    ("kötüleştir", True), # 'make it bad' — causative
    ("kötülük", True),    # 'badness' — nominal
]
example_wl = {"kötü"}  # minimal demo wordlist
for form, expected in toxic_root_examples:
    result = token_classifier(form, example_wl)
    hit = result["label"] == "toxic"
    print(f"'{form}':  token match={hit}  (should be {expected})  "
          f"-> {'OK' if hit == expected else 'MISSED'}")

## Section B — Embedding-based Toxicity Classifier (Per-language)

**Approach:**

1. Construct a training set by inserting words from the toxicity word list
   into simple sentence templates, creating contextually toxic examples.
   Safe examples are everyday benign sentences.
2. Obtain NLLB-200 embeddings for each training sentence.
3. Train a logistic regression classifier on the embeddings.
4. At inference time, embed the input sentence and classify.

Because the embeddings capture **morphological and semantic context** rather
than surface tokens, the classifier can recognise toxic intent even when
the exact word list entry is not present verbatim.

In [ ]:
# ---------------------------------------------------------------------------
# Construct training data from word list + templates
# ---------------------------------------------------------------------------
SAFE_TEMPLATES = [
    "Bugün hava çok güzel ve parkta yürüyüş yaptım.",
    "Toplantı çok verimli geçti, ekip harika çalıştı.",
    "Bu kitabı okumak çok keyifliydi.",
    "Arkadaşlarımla güzel bir akşam geçirdik.",
    "Yemek çok lezzetliydi, restoran mükemmeldi.",
    "Çocuklar bahçede mutlu bir şekilde oynuyor.",
    "Film gerçekten etkileyici ve düşündürücüydü.",
    "İş yerimiz çok verimli bir ortama sahip.",
    "Tatil boyunca harika anılar biriktirdik.",
    "Sınav sonuçları beklediğimden çok daha iyiydi.",
]

TOXIC_TEMPLATES = [
    "Sen tam bir {}!",
    "Bu {} bir insan.",
    "Onlar {} insanlardır.",
    "Ne kadar {} birisin.",
    "Böyle {} biriyle konuşmak istemiyorum.",
]

def build_training_data(wordlist: set, n_toxic: int = 30):
    """Sample n_toxic words from the word list and inject into templates."""
    import random
    words   = random.sample(list(wordlist), min(n_toxic, len(wordlist)))
    toxic_X, toxic_y = [], []
    for w in words:
        tmpl = random.choice(TOXIC_TEMPLATES)
        toxic_X.append(tmpl.format(w))
        toxic_y.append(1)
    safe_X  = SAFE_TEMPLATES * (max(1, n_toxic // len(SAFE_TEMPLATES)))
    safe_y  = [0] * len(safe_X)
    return toxic_X + safe_X, toxic_y + safe_y

# --- Train per-language embedding classifier for Turkish ---
turkicnlp.download("tur", processors=["embeddings"])
tur_embed = Pipeline("tur", processors=["embeddings"])

if turkish_wl:
    X_texts, y_tox = build_training_data(turkish_wl, n_toxic=40)
    print(f"Training set: {y_tox.count(1)} toxic, {y_tox.count(0)} safe")

    print("Embedding training sentences...")
    X_emb = [tur_embed(t).embedding for t in X_texts]

    clf_tox = LogisticRegression(max_iter=1000, random_state=42)
    clf_tox.fit(X_emb, y_tox)
    print("Classifier trained.")
else:
    print("Word list not available; run Section A.1 to download.")
    clf_tox = None

In [ ]:
# --- Inference: morphological variants caught by embedding approach ---
if clf_tox:
    test_cases = [
        ("Bugün hava çok güzel.", "safe"),
        ("Toplantı çok verimli geçti.", "safe"),
        ("Sen gerçekten kötüsün.", "ambiguous-neg"),
        ("Bu film harika, izleyin!", "safe"),
    ]
    print(f"{'Sentence':<45} {'Pred':>6}  {'P(toxic)':>9}  {'Expected':>10}")
    print("-" * 75)
    for sent, expected in test_cases:
        emb  = tur_embed(sent).embedding
        prob = clf_tox.predict_proba([emb])[0]
        pred = "toxic" if prob[1] >= 0.5 else "safe"
        print(f"{sent[:44]:<45} {pred:>6}  {prob[1]:>9.3f}  {expected:>10}")

## Section C — Unified Multilingual Toxicity Classifier

**Approach:**

Because NLLB-200 embeddings are language-agnostic, we can train a **single
logistic regression model** on combined training data from all nine FLORES
Turkic languages simultaneously. This model then works for all languages
at inference time, including those that contributed no or few training examples.

Steps:
1. For each language, build training sentences using the word list and templates.
2. Embed all sentences using the language-specific TurkicNLP pipeline.
3. Concatenate all embeddings into one large training matrix.
4. Train a single logistic regression.
5. Evaluate per-language accuracy on held-out test sentences.

In [ ]:
for iso, _, _ in FLORES_LANGS:
    turkicnlp.download(iso, processors=["embeddings"])

# Safe sentences translated to each language for diverse safe training data
SAFE_EN = [
    "Today the weather is very nice and I went for a walk in the park.",
    "The meeting was very productive, the team worked great.",
    "This book was very enjoyable to read.",
    "We spent a great evening with friends.",
    "The food was delicious and the restaurant was excellent.",
]

# Build combined multilingual training data
all_X, all_y = [], []

for iso, nllb_code, name in FLORES_LANGS:
    wl = wordlists.get(nllb_code, set())
    if not wl:
        print(f"  {name}: word list missing, skipping.")
        continue

    pipe = Pipeline(iso, processors=["embeddings"])
    trans_pipe = Pipeline("tur", processors=["translate"],
                          translate_tgt_lang=nllb_code)

    # Toxic examples
    import random
    words = random.sample(list(wl), min(20, len(wl)))
    for w in words:
        tmpl = random.choice(TOXIC_TEMPLATES)
        emb  = pipe(tmpl.format(w)).embedding
        all_X.append(emb)
        all_y.append(1)

    # Safe examples — translate generic English safe sentences
    for safe_en in SAFE_EN:
        safe_tgt = trans_pipe(safe_en).translation
        emb = pipe(safe_tgt).embedding
        all_X.append(emb)
        all_y.append(0)

    print(f"  {name:<15}: +{min(20,len(wl))} toxic, +{len(SAFE_EN)} safe")

print(f"\nTotal training samples: {len(all_X)}"
      f" ({all_y.count(1)} toxic, {all_y.count(0)} safe)")

In [ ]:
if len(all_X) > 10:
    X_tr, X_te, y_tr, y_te = train_test_split(
        all_X, all_y, test_size=0.2, random_state=42, stratify=all_y)

    clf_multi = LogisticRegression(max_iter=2000, random_state=42)
    clf_multi.fit(X_tr, y_tr)

    print("Overall evaluation on held-out multilingual test set:")
    print(classification_report(y_te, clf_multi.predict(X_te),
                                 target_names=["Safe", "Toxic"]))
else:
    print("Not enough data to train. Download word lists first.")
    clf_multi = None

In [ ]:
# Per-language accuracy on a small held-out set per language
if clf_multi:
    print(f"{'Language':<15} {'Acc (toxic)':>12} {'Acc (safe)':>11}")
    print("-" * 40)
    for iso, nllb_code, name in FLORES_LANGS:
        wl = wordlists.get(nllb_code, set())
        if not wl:
            continue
        pipe = Pipeline(iso, processors=["embeddings"])
        trans_pipe = Pipeline("tur", processors=["translate"],
                              translate_tgt_lang=nllb_code)

        # 5 toxic + 5 safe test sentences
        test_words = random.sample(list(wl), min(5, len(wl)))
        toxic_embs = [pipe(random.choice(TOXIC_TEMPLATES).format(w)).embedding
                      for w in test_words]
        safe_embs  = [pipe(trans_pipe(s).translation).embedding
                      for s in SAFE_EN[:5]]

        toxic_preds = clf_multi.predict(toxic_embs)
        safe_preds  = clf_multi.predict(safe_embs)
        acc_tox  = sum(p == 1 for p in toxic_preds) / len(toxic_preds)
        acc_safe = sum(p == 0 for p in safe_preds)  / len(safe_preds)
        print(f"{name:<15} {acc_tox:>12.0%} {acc_safe:>11.0%}")

## Summary

| Approach | Strengths | Recommended when |
|----------|-----------|------------------|
| Token-based | Zero training data needed, fully interpretable | Quick baseline; language has a good word list |
| Embedding (per-lang) | Context-aware, handles morphology | You have labelled examples for each target language |
| Unified multilingual | One model for all 9 languages, cross-lingual transfer | Low-resource languages; unified deployment |

For production systems, consider combining approaches: use the token list as a hard filter for known toxic terms, and the embedding classifier to catch novel or morphologically inflected forms.